In [62]:
from pymongo import MongoClient
from dotenv import load_dotenv
import os

load_dotenv()
collection_name = "articles_data"
mongo_uri = os.getenv("MONGO_URI_NAACP")
mongo_db_name = os.getenv("MONGO_DB_NAME_NAACP")
mongo_client = MongoClient(mongo_uri)

# Set up MongoDB connection
db_original = mongo_client[mongo_db_name]
locs_collection = db_original["locations_data"]
collection = db_original[collection_name]


In [58]:
# Aggregation pipeline to find the most common coordinates with associated locations
pipeline = [
    {
        "$unwind": {
            "path": "$coordinates",  # Unwind coordinates array
            "includeArrayIndex": "index"  # Capture the index of the unwound coordinate
        }
    },
    {
        "$group": {
            "_id": "$coordinates",  # Group by coordinate
            "count": { "$sum": 1 },  # Count occurrences
            "locations": { "$addToSet": { "$arrayElemAt": ["$locations", "$index"] } }  # Collect associated locations using the index
        }
    },
    {
        "$sort": { "count": -1 }  # Sort by count in descending order
    }
]

# Execute the aggregation pipeline
common_coordinates = collection.aggregate(pipeline)

# main_entries = []
# for entry in common_coordinates:
#     if entry['count'] > 200: # and entry['count'] < 270:
#         main_entries.append(entry)

# print(f"Found {len(main_entries)} entries with more than 100 occurrences")
# print("-" * 50)
# for entry in main_entries:
#     print(f"Coordinates: {entry['_id']}")
#     print(f"Count: {entry['count']}")
#     print(f"Associated Locations: {entry['locations']}")
#     print("-" * 50) 


In [59]:
all_locations = [entry for entry in common_coordinates]
len(all_locations)

2738

In [46]:
print(len(all_locations))

location_count = 0
for entry in all_locations:
    location_count += len(entry['locations'])
print(location_count)

2738
4645


In [50]:
repeated_locations = []
repeated_locations_count = 0
for entry in all_locations:
    if len(entry['locations']) > 1:
        repeated_locations.append(entry)
        repeated_locations_count += len(entry['locations'])


# for entry in repeated_locations:
#     print(f"Coordinates: {entry['_id']}")
#     print(f"Count: {entry['count']}")
#     print(f"Associated Locations: {entry['locations']}")
#     print("-" * 50)

In [51]:
print(repeated_locations_count)
print(len(repeated_locations))

2584
677


In [74]:
collection_name = "articles_data"
mongo_uri = os.getenv("MONGO_URI_NAACP")
mongo_db_name = os.getenv("MONGO_DB_NAME_NAACP")
mongo_client = MongoClient(mongo_uri)

# Set up MongoDB connection
db = mongo_client["locations_test"]
locations_collection = db["locations_data"]

In [61]:
for entry in all_locations:
    entry["all_locations"] = entry["locations"]
    del entry["locations"]
    entry["coordinates"] = entry["_id"]
    del entry["_id"]
    locations = entry["all_locations"]
    if len(locations) > 1:
        possible_locations = []
        for location in locations:
            if len(location) > 3:
                possible_locations.append(location)

        locations = possible_locations
        
        if len(locations) < 2:
            entry["location"] = locations[0]
            continue

        sorted_locations = sorted(locations, key=len)

        def find_substring(sorted_list):
            for i in range(len(sorted_list)):
                substring = sorted_list[i]
                # Check if this substring is in at least some of the others
                count = sum(1 for other in sorted_list if substring in other and other != substring)
                if count > 0:
                    return substring
            return None

        # Get the result
        result = find_substring(sorted_locations)
        if result:
            entry["location"] = result
        else:
            if len(sorted_locations[0]) > 1:
                entry["location"] = sorted_locations[1]
            else:
                entry["location"] = sorted_locations[0]
    else:
        entry["location"] = locations[0]
    
    locations_collection.insert_one(entry)
    
    print(entry)
    

{'count': 397, 'all_locations': ['american federation of teachers', 'american federation of teachers massachusetts', 'aft'], 'coordinates': [-71.0592776, 42.3535523], 'location': 'american federation of teachers', '_id': ObjectId('66ecac6444cdb452a68cf9e1')}
{'count': 327, 'all_locations': ['new york city health department', 'new york city ballet', 'new york metropolitan transportation authority', 'wildlife conservation society new york aquarium', 'nbc new york', 'new york cosmos', 'new york city fc', 'new york philharmonic', 'new york theatre workshop', 'new york metropolitan museum of art', 'new york post', 'new york memorial sloan kettering cancer center', 'new york city police department hate crimes task force', 'early music new york', 'new york city pride', 'new york city mayors office', 'new york time', 'new york museum of play', 'now nyc', 'city university of new york', 'new york rangers', 'new york police department', 'new york times', 'new york city transit', 'new york city fi

In [63]:
import os
import pandas as pd

import requests
import googlemaps

from geopy.distance import distance
from geopy.distance import geodesic

from dotenv import load_dotenv

from tqdm import tqdm

# Load environment variables
load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [64]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}", components={"country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']

            # Get the city from the address components
            address_components = geocode_result[0]['address_components']
            city = None
            for component in address_components:
                if 'locality' in component['types']:
                    city = component['long_name']
                    break
            
            return [latitude, longitude], city
        else:
            print(f"[WARNING] Could not find location {location} through google maps")
            return [None, None], None
    except Exception as error:
        print(f"[ERROR] Error finding locations through google maps, {error}")
        return [None, None], None

In [77]:
locations_collection = db["locations_data_2"]

truly_all_locations = locs_collection.find()
for entry in truly_all_locations:
    del entry["_id"]
    coords, city = callGoogleMapsAPI(entry['value'])
    coords = [coords[1], coords[0]]
    old_coords = entry['coordinates']

    entry["old_coordinates"] = old_coords
    entry["coordinates"] = coords
    
    entry["old_city"] = entry["city"]
    entry["city"] = city

    if coords[0] is not None and old_coords[0] is not None:
        distance_km = distance((coords[1], coords[0]), (old_coords[1], old_coords[0])).km
    else:
        distance_km = None

    entry["distance_km"] = distance_km
    entry["all_locations"] = [entry["value"]]

    same_loc_doc = locations_collection.find_one({"coordinates": coords})
    if same_loc_doc:
        same_loc_doc["all_locations"].append(entry["value"])
        locations_collection.update_one({"coordinates": coords}, {"$set": {"all_locations": same_loc_doc["all_locations"]}})
    else:
        locations_collection.insert_one(entry)
        


[WARNING] Could not find location laredo colombia solidarity international bridge through google maps
[WARNING] Could not find location blue ledge through google maps
[WARNING] Could not find location plymouth african methodist episcopal church through google maps


KeyError: 'coordinates'

In [79]:
all_locs = locations_collection.find()
for entry in all_locs:
    locations = entry["all_locations"]
    if len(locations) > 1:
        possible_locations = []
        for location in locations:
            if len(location) > 3:
                possible_locations.append(location)

        locations = possible_locations
        
        if len(locations) < 2:
            entry["location"] = locations[0]
            continue

        sorted_locations = sorted(locations, key=len)

        def find_substring(sorted_list):
            for i in range(len(sorted_list)):
                substring = sorted_list[i]
                # Check if this substring is in at least some of the others
                count = sum(1 for other in sorted_list if substring in other and other != substring)
                if count > 0:
                    return substring
            return None

        # Get the result
        result = find_substring(sorted_locations)
        if result:
            entry["location"] = result
        else:
            if len(sorted_locations[0]) > 1:
                entry["location"] = sorted_locations[1]
            else:
                entry["location"] = sorted_locations[0]
    else:
        entry["location"] = locations[0]
    
    locations_collection.update_one({"coordinates": entry["coordinates"]}, {"$set": {"value": entry["location"]}})
        

In [80]:
all_locs = locations_collection.find()
for entry in all_locs:
    if entry["city"] is None:
        print(entry)
        print("-" * 50)

    if entry["distance_km"] > 1:
        print(entry)
        print("-" * 50)        


{'_id': ObjectId('66ecb98a44cdb452a68d0497'), 'value': 'travis county democratic party', 'articles': ['cb2e3904ab0e2dd683947efb5c8491b18a0e2faa0f9a798da8655769ca6c314d'], 'city': None, 'coordinates': [-97.69822719999999, 30.2097015], 'old_coordinates': [-97.69822719999999, 30.2097015], 'old_city': None, 'distance_km': 0.0, 'all_locations': ['travis county democratic party', 'travis county democratic party']}
--------------------------------------------------
{'_id': ObjectId('66ecb98a44cdb452a68d0498'), 'value': 'texas rangers', 'articles': ['cb2e3904ab0e2dd683947efb5c8491b18a0e2faa0f9a798da8655769ca6c314d'], 'city': None, 'coordinates': [-99.9018131, 31.9685988], 'old_coordinates': [-99.9018131, 31.9685988], 'old_city': None, 'distance_km': 0.0, 'all_locations': ['texas the republican party', 'texas the republican party', 'texas supreme court', 'texas department of public safety', 'texas methodist hospital', 'texas am transportation institute', 'texas u.s. house of representatives', '

TypeError: '>' not supported between instances of 'NoneType' and 'int'

In [7]:
print(filtered_df)

Empty DataFrame
Columns: []
Index: []


In [6]:
print(tabulate(filtered_df, headers='keys', tablefmt='psql'))

In [53]:
target_coordinate = [-77.0059094, 38.9037551]
# [-71.0588305, 42.3600709], [-71.0286605, 42.2806544], [-70.9078346, 42.3020647], [-70.9494938, 42.46676300000001], [-71.58990539999999, 42.5611947], [-71.1727583, 42.3328668], [-71.31617179999999, 42.6334247]
# [-71.0561981, 42.3587684] Boston Globe
# [-74.0059728, 40.7127753] New York City related
# [-71.0854325, 42.3284483] Boston Public Schools
# [-72.1553494, 42.5725316] gbh 2?
# [-71.8022934, 42.2625932] Worcester
# [-71.10973349999999, 42.3736158] Cambridge
# [-70.8954626, 42.5197473] Salem
# [-71.06485289999999, 42.29948479999999] Dorchester
# [-122.4194155, 37.7749295] San Fransisco
# [-87.6297982, 41.8781136] Chicago
# 
documents = collection.find({"coordinates": target_coordinate})

In [54]:
matching_count = collection.count_documents({"coordinates": target_coordinate})
print(f"Number of matching documents: {matching_count}")

Number of matching documents: 23


In [50]:
locations_collection = db["locations_data"]

result = locations_collection.delete_many({"coordinates": target_coordinate})
print(f"Number of documents deleted: {result.deleted_count}")

Number of documents deleted: 1


In [ ]:
for doc in documents:
    index = next((i for i, coord in enumerate(doc['coordinates']) if coord == target_coordinate), -1)
    
    if index > -1:
        new_coordinates = [coord for i, coord in enumerate(doc['coordinates']) if i != index]
        new_location = [loc for i, loc in enumerate(doc['locations']) if i != index]
        new_tract = [tract for i, tract in enumerate(doc['tracts']) if i != index]
        new_county = [county for i, county in enumerate(doc['counties']) if i != index]
        new_neighborhood = [neighborhood for i, neighborhood in enumerate(doc['neighborhoods']) if i != index]

        # Prepare the update operation
        update = {
            "$set": {
                "coordinates": new_coordinates,
                "locations": new_location,
                "tracts": new_tract,
                "counties": new_county,
                "neighborhoods": new_neighborhood
            }
        }
        
        # Apply the update operation
        collection.update_one({"_id": doc["_id"]}, update)
        print(f"Updated document with _id: {doc['_id']}")


print("Update operation completed.")

In [ ]:
target_coordinates = [[-77.0059094, 38.9037551], [-71.0561981, 42.3587684], [-71.0588305, 42.3600709], [-71.0286605, 42.2806544], [-70.9078346, 42.3020647], [-70.9494938, 42.46676300000001], [-71.58990539999999, 42.5611947], [-71.1727583, 42.3328668], [-71.31617179999999, 42.6334247]]

for target_coordinate in target_coordinates:
    documents = collection.find({"coordinates": target_coordinate})

    matching_count = collection.count_documents({"coordinates": target_coordinate})
    print(f"Number of matching documents: {matching_count}")

    locations_collection = db["locations_data"]
    result = locations_collection.delete_many({"coordinates": target_coordinate})
    print(f"Number of documents deleted: {result.deleted_count}")

    for doc in documents:
        index = next((i for i, coord in enumerate(doc['coordinates']) if coord == target_coordinate), -1)
        
        if index > -1:
            new_coordinates = [coord for i, coord in enumerate(doc['coordinates']) if i != index]
            new_location = [loc for i, loc in enumerate(doc['locations']) if i != index]
            new_tract = [tract for i, tract in enumerate(doc['tracts']) if i != index]
            new_county = [county for i, county in enumerate(doc['counties']) if i != index]
            new_neighborhood = [neighborhood for i, neighborhood in enumerate(doc['neighborhoods']) if i != index]
            new_state = [state for i, state in enumerate(doc['states']) if i != index]
            new_city = [city for i, city in enumerate(doc['cities']) if i != index]

            # Prepare the update operation
            update = {
                "$set": {
                    "coordinates": new_coordinates,
                    "locations": new_location,
                    "tracts": new_tract,
                    "counties": new_county,
                    "neighborhoods": new_neighborhood,
                    "states": new_state,   # Include missing 'states'
                    "cities": new_city     # Include missing 'cities'
                }
            }
            
            # Apply the update operation
            collection.update_one({"_id": doc["_id"]}, update)
            print(f"Updated document with _id: {doc['_id']}")


print("Update operation completed.")
    
    
    

In [ ]:
tracts_data_collection = db["tracts_data"]
count = 0
update_count = 0
for doc in collection.find():
    # Check if lengths are consistent
    
    length_check = len(doc['coordinates']) == len(doc['locations']) == len(doc['tracts']) == len(doc['counties']) == len(doc['neighborhoods']) == len(doc['states']) == len(doc['cities'])
    
    if not length_check:
        # Iterate through each article document
        cities = doc['cities']
        states = doc['states']

        # Iterate through each tract in the document
        for i, tract in enumerate(doc['tracts']):
            # Get the expected city and state for the current tract from tracts_data collection
            tracts_data_doc = tracts_data_collection.find_one({"tract": tract})
            
            if tracts_data_doc:
                expected_city = tracts_data_doc.get('city', None)
                expected_state = tracts_data_doc.get('state', None)
                
                current_city = doc['cities'][i]
                current_state = doc['states'][i]
                
                if current_city != expected_city or current_state != expected_state:
                    new_cities = [cities for n, cities in enumerate(doc['cities']) if n != i]
                    new_states = [states for n, states in enumerate(doc['states']) if n != i]

                    # If mismatches were found, update the document by setting the new lists
                    collection.update_one(
                        {"_id": doc["_id"]}, 
                        {"$set": {
                            "cities": new_cities,
                            "states": new_states
                        }}
                    )
                    count += 1
                    break
        
    print(f"Updated document with _id: {doc['_id']}")

# Summary of corrections
print(f"Total documents updated: {count}")
print(f"Total updates made: {update_count}")
                


In [ ]:
count2 = 0
for doc in collection.find():
    # Check if lengths are consistent
    ids = []
    length_check = len(doc['coordinates']) == len(doc['locations']) == len(doc['tracts']) == len(doc['counties']) == len(doc['neighborhoods']) == len(doc['states']) == len(doc['cities'])
    if not length_check:
        count2 += 1
        ids.append(doc['_id'])

print(f"Total documents with inconsistent lengths: {count2}")
print(ids)

In [29]:
# Criteria to find documents with no locations or an empty locations list
query = {"$or": [{"locations": {"$exists": False}}, {"locations": []}]}

for doc in collection.find(query):

    update = {
        "$set": {
            "cities": [],
            "states": []
        }
    }

    result = collection.update_one({"_id": doc["_id"]}, update)
    print(f"Updated document with _id: {doc['_id']}")

print("Update operation completed.")

Updated document with _id: 14cedf4359e33bed670150a3a57f0e7d0c992eea8d0bd511b2d7055b5daff90f
Updated document with _id: c48889b8bc1281f13469898c4778cb4c4bf9fae8aee2f28515dd0988c50a7f02
Updated document with _id: a200f3967b291e9c5d2c47064d46571dccc55d4e3cf21b7c7ad2aeb616ba2a38
Updated document with _id: 0f1d2254121cb70742dcf7c38afd26051a37286daa58761d96783d458f2a2d02
Updated document with _id: 7a4119e98c03fd21b41d3cfc8b0ef644f9c08affd3bc0017797e2bed4d19270c
Updated document with _id: c5394d7fd7c6c78defc9451eff1e84b3fa52feffd08958af43c2ce7d1a4091f1
Updated document with _id: 4413e86d84d7a0ef69e62f45b73119400789a8e10f3bdb7203029a6dd377441e
Updated document with _id: ada43546c112c172216bad0a798bbf77c42233e370dd58271d0b2e7e034eb066
Updated document with _id: 6427dadcc0688786bc6669a6640fd444eb77eca4d94c57672c7951a65326d97f
Updated document with _id: 40c66e7d078b2c3d3181a7c30575a3434625cfb92eec229a68aafedf91176fdf
Updated document with _id: fa129a838a25a261eb5d862b6a28321c381d5ac658a8733c56ce1

In [36]:
matching_count = collection.count_documents({"coordinates": target_coordinate})
print(f"Number of matching documents: {matching_count}")

Number of matching documents: 0


In [2]:
from tqdm import tqdm

source_collection = db["articles_data"]
location_collection = db["locations_data"] 

documents = list(source_collection.find())

for doc in tqdm(documents, desc="Processing documents"):
    locations = doc['locations']
    coordinates = doc['coordinates']

    # Ensure lists are of the same length
    if len(locations) != len(coordinates):
        continue 

    for i, location in enumerate(locations):
        coordinate = coordinates[i]
        
        location_doc = location_collection.find_one({"value": location})
        
        if location_doc:
            if 'coordinates' not in location_doc or location_doc['coordinates'] != coordinate:
                # If coordinate is not in the list, update the document by adding it
                location_collection.update_one(
                    {"value": location},
                    {"$set": {"coordinates": coordinate}}
                )
        else:
            location_collection.insert_one({
                "value": location,
                "coordinates": [coordinate]
            })
        

print("Coordinates have been added to the locations collection.")

Processing documents: 100%|██████████| 10475/10475 [21:58<00:00,  7.94it/s]

Coordinates have been added to the locations collection.


In [ ]:
location_collection = db["locations_data"] 

locations = list(location_collection.find())


